# Customers: Silver to Gold aggregations

Subscriptions per country and subscriptions in 2020 & 2021, computed from the pipeline's `customers_silver` dataset.

In [0]:
# Gold: subscriptions count per country (from the silver dataset)
from pyspark import pipelines as dp
from pyspark.sql.functions import col, year

@dp.materialized_view(name="customers_gold", comment="Subscriptions count per country")
def customers_gold_country():
    df_silver = spark.read.table("customers_silver")

    return (
        df_silver
        .groupBy("country")
        .count()
        .withColumnRenamed("count", "subscriptions_count")
        .orderBy(col("subscriptions_count").desc())
    )



In [0]:
# Gold: subscriptions count in 2020 and 2021
@dp.materialized_view(name="customers_years", comment="Subscriptions count in 2020 and 2021")
def customers_gold_years():
    df_silver = spark.read.table("customers_silver")

    return (
        df_silver
        .filter(year("subscription_date").isin(2020, 2021))
        .groupBy(year("subscription_date").alias("subscription_year"))
        .count()
        .withColumnRenamed("count", "subscriptions_count")
        .orderBy("subscription_year")
    )



In [0]:
# Gold: subscriptions count for every country, pivoted by subscription year
import re

@dp.materialized_view(name="customers_gold_pivot", comment="Subscriptions count per country, pivoted by subscription year")
def customers_gold_every_country_sub_num():
    df_silver = spark.read.table("customers_silver")

    pivoted = (
        df_silver
        .groupBy(year("subscription_date").alias("subscription_year"))
        .pivot("country")
        .count()
    )

    # Country names (e.g. "United Kingdom") contain characters that are invalid
    # in Delta column names, so sanitize the pivoted column names.
    clean_columns = [re.sub(r"[ ,;{}()\n\t=]", "_", c.strip()).lower() for c in pivoted.columns]

    return pivoted.toDF(*clean_columns).orderBy("subscription_year")
